# Classification de texte avec des SVMs

Dans ces travaux pratiques, nous allons utiliser des SVMs pour retrouver l'auteur d'un texte.

Le corpus utilisé est tiré d'œuvres littéraires classiques de littérature anglophone.

9 auteurs, 2 livres par auteur avec un fichier par chapitre.

Dans un premier temps, récupérons ce corpus de texte :

In [ ]:
spacy_updated = !pip freeze | grep spacy==3
if not spacy_updated:
  !pip install 'spacy >= 3, < 4'

In [ ]:
! git clone https://github.com/nzmonzmp/dataset-9classical-author.git

In [ ]:
import pathlib
import re
import time
import typing

import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.feature_extraction.text
import sklearn.preprocessing
import sklearn.model_selection
import sklearn.svm
import spacy
import tqdm.notebook

## Création de la base d'apprentissage

Après un rapide coup d'oeil aux données du corpus, implémentez la fonction `get_data` qui :
- prend en argument le chemin du corpus
- renvoie deux listes :
  - la première contient les textes
  - la seconde l'auteur correspondant

In [ ]:
def get_data(directory: pathlib.Path
             ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors = []

  # Votre code ici

  return texts, authors


texts, authors = get_data(pathlib.Path("dataset-9classical-author"))

### Solution

In [ ]:
def get_data(directory: pathlib.Path
             ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors = []
  for item in tqdm.notebook.tqdm(list(directory.glob("*/*/*.txt"))):
    # Récupération de l'auteur ou autrice
    # Avec parent
    author = item.parent.parent.name
    # Avec parts
    author = item.parts[1]

    # Récupération du texte
    # Avec open
    with item.open(encoding="utf8") as fh:
      text = fh.read()
    # Avec read_text
    text = item.read_text(encoding="utf8")

    # Ajout aux listes
    texts.append(text)
    authors.append(author)

  return texts, authors


texts, authors = get_data(pathlib.Path("dataset-9classical-author"))

## Représentation TF-IDF

Utilisez l'objet `TfidfVectorizer` de la librairie `sklearn` afin de transformer `texts` pour que nos données textuelles puissent être traitées par un modèle SVM. Stockez le résultat dans la variable `X`.

In [ ]:
# Votre code ici

## Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(texts)

## Label Encoding

À l'aide de l'objet `LabelEncoder` de `sklearn`, procédez au label encoding de `authors` pour qu'ils puissent être traité par un SVM. Stockez le résultat dans la variable `y`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
# En python de base
index_to_author = list(set(authors))
author_to_index = {author: index
                   for index, author
                   in enumerate(index_to_author)}
y = numpy.array([author_to_index[author] for author in authors])

# Avec scikit-learn
le = sklearn.preprocessing.LabelEncoder()
y = le.fit_transform(authors)

## Apprentissage et évaluation

Utilisez la fonction `cross_val_score` de `sklearn` pour évaluer un modèle SVM.

- Utilisez le modèle `sklearn.svm.SVC` avec un kernel linéaire
- Faites 5 fold de cross validation
- À l'aide du module `time` (et de sa fonction `time`) de la bibliothèque standard, affichez le temps nécessaire aux 5 fold

In [ ]:
# Votre code ici

### Solution

In [ ]:
start_time = time.time()
clf = sklearn.svm.SVC(kernel='linear')
scores = sklearn.model_selection.cross_val_score(clf, X, y, cv=5)
print(f"L'entraînement a pris {time.time() - start_time:.2f} secondes")
print(scores)

## Optimisation quand on utilise un kernel linéaire

Réutilisez le même code que précédemment mais maintenant utilisez ``sklearn.svm.LinearSVC`` comme modèle. Pour faire simple, c'est une implémentation rapide du modèle précédent.

In [ ]:
# Votre code ici

### Solution

In [ ]:
start_time = time.time()
clf = sklearn.svm.LinearSVC()
scores = sklearn.model_selection.cross_val_score(clf, X, y, cv=5)
print(f"L'entraînement a pris {time.time() - start_time:.2f} secondes")
print(scores)

## Conclusions

Que peut-on conclure sur les scores obtenus ?

### Solution

La performance extrèmement proche de 1 est suspecte. On pourrait par exemple formuler deux hypothèses quant à ce score très haut :

- la taille des extraits (chapitres) est telle qu'il est facile pour le modèle de repérer au moins quelques mots qui sont spécifiques à l'auteur ou l'autrice.
- peut être est-ce le fait que des noms de personnage ou de lieu sont cités dans presque tous les chapitres qui permet au modèle d'atteindre ce résultat.

Nous pouvons enviser une méthode pour tester chacune des hypothèses :

- entraîner au niveau de la phrase pour tester l'impact de la taille
- entraîner sans les entités nommées pour tester l'impact des noms de personnages et de lieux

## Remplacement des entités nommées par des marqueurs spéciaux

Pour remplacer les noms de personnages et de lieux (et plus généralement toutes les entités nommées), nous allons utiliser `spacy`, que nous devons charger. Nous n'aurons pas besoin de la plupart des composants de la pipeline `spacy` anglais que nous récupérerons. Nous allons donc les désactiver et activer le seul composant supplémentaire dont nous aurons besoin (le délimiteur de phrases).

In [ ]:
#!python -m spacy download en_core_web_lg
!python -m spacy download en_core_web_sm

In [ ]:
nlp = spacy.load(
    "en_core_web_sm",
    exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"))
nlp.enable_pipe("senter")

Pour tester nos hypothèses quant au sur-apprentissage de notre précédent modèle, commençons par définir une fonction pour remplacer les entités nommées par leur type.

In [ ]:
doc = nlp("Jimmy Hendrix was absolutely amazing at Woodstock!")

# Affichage des entités nommées
print(" ".join(f"{t}/{t.ent_type_}/{t.ent_iob_}" if t.ent_type_ else t.text
               for t in doc))

Par exemple, si le rendu ci-dessus est :

    Jimmy/PERSON/B Hendrix/PERSON/I was absolutely amazing at Woodstock/ORG/B !

On souhaite coder la fonction `replace_ners` pour qu'elle retourne :

    PPPERSON was absolutely amazing at OOORG !

Pour cela, utilisez les [attributs `ent_type_` & `ent_iob_`](https://spacy.io/api/token#attributes) des tokens obtenus avec le modèle `spacy` (comme vous pouvez le constater, on triple la première lettre : c'est une tactique pour éviter d'utiliser un mot déjà dans le vocabulaire).

In [ ]:
def replace_ners(tokens: typing.Iterable[spacy.tokens.Token]) -> str:
  # Votre code ici
  return ""


replace_ners(doc)

### Solution

In [ ]:
def replace_ners(tokens: typing.Iterable[spacy.tokens.Token]) -> str:
  result = []
  for token in tokens:
    if token.ent_iob_ == "O":
      result.append(token.text)
    elif token.ent_iob_ == "B":
      result.append(f"{token.ent_type_[0] * 2}{token.ent_type_}")
  return " ".join(result)


print(replace_ners(doc))

## Fonction `get_data` étendue

Voici la fonction `get_data` retravaillée pour qu'elle renvoie les auteurs et textes aux deux échelles de la phrase et du document, ainsi qu'avec ou sans remplacement des NERs pour les textes.

Elle renvoie un objet qui contient 6 champs qui correspondent aux 4 versions des textes et 2 versions des auteurs.

In [ ]:
class Data(typing.NamedTuple):
  texts_sentence_replaced_ners: typing.List[str]
  texts_sentence_normal: typing.List[str]
  texts_document_replaced_ners: typing.List[str]
  texts_document_normal: typing.List[str]
  authors_sentence: typing.List[str]
  authors_document: typing.List[str]


def get_data(directory: pathlib.Path,
             ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors_sentence = []
  authors_document = []
  texts_sentence_replaced_ners = []
  texts_sentence_normal = []
  texts_document_replaced_ners = []
  texts_document_normal = []
  paths = list(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.notebook.tqdm(zip(paths, nlp.pipe(contents,
                                                          n_process=-1)),
                                      total=len(paths)):

    author = path.parent.parent.name
    for sentence in doc.sents:
      authors_sentence.append(author)
      texts_sentence_replaced_ners.append(replace_ners(sentence))
      texts_sentence_normal.append(" ".join(token.text for token in sentence))
    authors_document.append(author)
    texts_document_replaced_ners.append(replace_ners(doc))
    texts_document_normal.append(" ".join(token.text for token in doc))
  return Data(authors_sentence=authors_sentence,
              authors_document=authors_document,
              texts_sentence_replaced_ners=texts_sentence_replaced_ners,
              texts_sentence_normal=texts_sentence_normal,
              texts_document_replaced_ners=texts_document_replaced_ners,
              texts_document_normal=texts_document_normal)


data = get_data(pathlib.Path("dataset-9classical-author"))

## Apprentissage et évaluation

Quelles sont maintenant les performances d'une 5-fold cross validation pour les différentes configurations ?

Conseil : créez une fonction qui regroupe les prétraitements à effectuer et qui retourne la moyenne des scores obtenus par `cross_val_score` pour des textes donnés et les auteurs correspondants.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def evaluate(texts: typing.List[str], authors: typing.List[str]) -> float:
  vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
  X = vectorizer.fit_transform(texts)

  le = sklearn.preprocessing.LabelEncoder()
  y = le.fit_transform(authors)

  clf = sklearn.svm.LinearSVC()
  scores = sklearn.model_selection.cross_val_score(clf, X, y, cv=5)
  return scores.mean()


print("À l'échelle du document,  sans remplacement des NERs :",
      evaluate(data.texts_document_normal, data.authors_document))
print("À l'échelle du document,  avec remplacement des NERs :",
      evaluate(data.texts_document_replaced_ners, data.authors_document))
# print("À l'échelle de la phrase, sans remplacement des NERs :",
#       evaluate(data.texts_sentence_normal, data.authors_sentence))
# print("À l'échelle du la phrase, avec remplacement des NERs :",
#       evaluate(data.texts_sentence_replaced_ners, data.authors_sentence))

## Affichage de la matrice de confusion en Test

Choisissez une configuration de textes (échelle document ou phrase, avec ou sans remplacement des NERs), puis :

- Obtenez `X` et `y` à partir des textes choisis et auteurs correspondants
- Séparez `X` et `y` en bases d'apprentissage et de test à l'aide de `train_test_split` de `sklearn`
- Entraînez un modèle sur la base d'apprentissage
- Évaluez les performances sur la base de test
- Affichez la matrice de confusion, `confusion_matrix` de `sklearn` (l'utilisation d'une heatmap est recommandée : `heatmap` de `seaborn`)

In [ ]:
# Votre code ici

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(data.texts_document_replaced_ners)

le = sklearn.preprocessing.LabelEncoder()
y = le.fit_transform(data.authors_document)

In [ ]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.3, random_state=42)

In [ ]:
clf = sklearn.svm.LinearSVC()
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
y_test_pred = clf.predict(X_test)
conf_mat = sklearn.metrics.confusion_matrix(y_test, y_test_pred)
total = conf_mat.sum(axis=0)
with numpy.printoptions(precision=2, suppress=True):
  print(conf_mat / total)
print(total)

In [ ]:
seaborn.heatmap(conf_mat,
                cmap="rocket_r",
                xticklabels=le.classes_,
                yticklabels=le.classes_,
                annot=True,
                fmt="d")
plt.show()

## Analyse du modèle obtenu

`LinearSCV` utilise la stratégie un contre tous (`multi_class='ovr'`), notre modèle est donc constitué de 9 SVMs de classification binaire.

L'attribut `coef_` de notre objet `LinearSCV` contient 9 vecteurs de dimension égale à la taille de notre dictionnaire.

Dans le but de faire des prédictions, c'est avec ces vecteurs que nous calculons leurs produits scalaires avec nos documents représentées sous forme TF-IDF.

Chacun de ces 9 vecteurs correspond à l'auteur en suivant la liste `le.classes_`

Par convention d'implémentation, la classe positive correspond à la classe seule (stratégie un contre tous)

A l'aide de `numpy.argsort`, on peut ainsi afficher les mots les plus discriminants pour chaque auteur quand ils sont face à tous les autres auteurs.



In [ ]:
n_best = 10
for i, author in enumerate(le.classes_):
  author_vec = clf.coef_[i]
  index = numpy.argsort(-author_vec)[:n_best]
  top_words = vectorizer.get_feature_names_out()[index]
  top_scores = author_vec[index]
  bestwords = (f"{score:.2f} {word:<15}"
               for score, word in zip(top_scores, top_words))
  print(f"{author:<15} {' '.join(bestwords)}|")